Prueba

In [1]:
pip install hyperopt

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from keras.models import Sequential
from keras.layers import Dense, Dropout, GRU, InputLayer, Flatten
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import os
np.random.seed(42)


LEER DATASET

In [3]:
import pandas as pd

datos = pd.read_csv("C:\\Users\\wamt1\\Desktop\\pruebas_collab\\SNpollutionFinal.csv")

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [4]:
datos.head()

,pollution,dew,temp,press,wnd_dir,wnd_spd
date,,,,,,
2010-01-01 00:00:00,129.0,-21,-11.0,1021.0,2,1.79
2010-01-01 01:00:00,129.0,-21,-12.0,1020.0,2,4.92
2010-01-01 02:00:00,129.0,-21,-11.0,1019.0,2,6.71
2010-01-01 03:00:00,129.0,-21,-14.0,1019.0,2,9.84
2010-01-01 04:00:00,129.0,-20,-12.0,1018.0,2,12.97


Se eliminan las primeras 24 filas debido a que estas contenían valores NAN en la columna pollution, y habían sido rellenadas con interpolación lineal.

In [5]:
datos = datos.drop(datos.index[:24])

datos

,pollution,dew,temp,press,wnd_dir,wnd_spd
date,,,,,,
2010-01-02 00:00:00,129.0,-16,-4.0,1020.0,1,1.79
2010-01-02 01:00:00,148.0,-15,-4.0,1020.0,1,2.68
2010-01-02 02:00:00,159.0,-11,-5.0,1021.0,1,3.57
2010-01-02 03:00:00,181.0,-7,-5.0,1022.0,1,5.36
2010-01-02 04:00:00,138.0,-7,-5.0,1022.0,1,6.25
...,...,...,...,...,...,...
2014-12-31 19:00:00,8.0,-23,-2.0,1034.0,2,231.97
2014-12-31 20:00:00,10.0,-22,-3.0,1034.0,2,237.78
2014-12-31 21:00:00,10.0,-22,-3.0,1034.0,2,242.70


In [6]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Se divide el dataset

In [7]:
# Dividir el conjunto de datos en entrenamiento y prueba
train, test = train_test_split(datos, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
test, val = train_test_split(test, test_size=0.33, shuffle=False)

print("Las dimensiones de train son: ", train.shape)
print("Las dimensiones de test son: ", test.shape)
print("Las dimensiones de val son: ", val.shape)

Las dimensiones de train son:  (30660, 6)
Las dimensiones de test son:  (8803, 6)
Las dimensiones de val son:  (4337, 6)


Se normalizan los datos

In [8]:
from sklearn.preprocessing import StandardScaler


# Normalizar solo con los datos de entrenamiento
scaler = StandardScaler()
train = scaler.fit_transform(train)

# Aplicar la transformación a test y val usando los parámetros de train
test = scaler.transform(test)
val = scaler.transform(val)

train = pd.DataFrame(train, columns=datos.columns)
test = pd.DataFrame(test, columns=datos.columns)
val = pd.DataFrame(val, columns=datos.columns)


Se unen los datos nuevamente, ahora normalizados, en un único conjunto.

In [9]:
datosNormalizados = pd.concat([train, test, val])

datosNormalizados.index = datos.index


In [10]:
datosNormalizados.shape

(43800, 6)

In [11]:
datosNormalizados.head(10)


,pollution,dew,temp,press,wnd_dir,wnd_spd
date,,,,,,
2010-01-02 00:00:00,0.317681,-1.214023,-1.268524,0.329687,-0.380944,-0.464048
2010-01-02 01:00:00,0.526152,-1.144302,-1.268524,0.329687,-0.380944,-0.446575
2010-01-02 02:00:00,0.646846,-0.865419,-1.349314,0.426127,-0.380944,-0.429103
2010-01-02 03:00:00,0.888234,-0.586536,-1.349314,0.522567,-0.380944,-0.393962
2010-01-02 04:00:00,0.416431,-0.586536,-1.349314,0.522567,-0.380944,-0.376489
2010-01-02 05:00:00,0.098238,-0.586536,-1.430104,0.522567,-0.380944,-0.359017
2010-01-02 06:00:00,0.054349,-0.586536,-1.430104,0.619008,-0.380944,-0.323876
2010-01-02 07:00:00,0.262820,-0.586536,-1.349314,0.715448,-0.380944,-0.288735
2010-01-02 08:00:00,0.218931,-0.656257,-1.430104,0.715448,-0.380944,-0.253594


Espacio de búsqueda

In [12]:
space = {
    'layers': hp.quniform('layers', 1, 4, 1), # Cantidad de capas GRU
    'units': hp.choice('units', [2 ** i for i in range(3, 8)]),  # Número de unidades GRU
    'activation': hp.choice('activation', ['tanh', 'sigmoid', 'relu', 'linear']),
    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'epochs': hp.choice('epochs', [2 ** i for i in range(3, 9)]),  # Número de épocas de entrenamiento
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)])
}

Se establece el formato de datos de entrada para redes GRU, es decir [observaciones, retardos, caracteristicas]

In [13]:
futuros = 24
pasados  = 12

In [14]:
datosX = []
datosY = []
for i in range(pasados, len(datosNormalizados) - futuros + 1):
  datosX.append(datosNormalizados.iloc[i-pasados:i, 0:datosNormalizados.shape[1]])
  datosY.append(datosNormalizados.iloc[i+futuros-1:i+futuros, 0])


In [15]:
# Convertir las listas en arrays numpy
datosX = np.array(datosX)
datosY = np.array(datosY)

# Ver las dimensiones (shape) de los arrays
print("Dimensiones de X:", datosX.shape)  # (n_muestras, pasos_de_tiempo, n_características)
print("Dimensiones de Y:", datosY.shape)  # (n_muestras, n_características)

Dimensiones de X: (43765, 12, 6)
Dimensiones de Y: (43765, 1)


In [16]:
print(datosX[0])

[[ 0.31768099 -1.2140229  -1.26852411  0.32968671 -0.38094383 -0.46404777]
 [ 0.52615226 -1.14430217 -1.26852411  0.32968671 -0.38094383 -0.44657536]
 [ 0.64684616 -0.86541928 -1.34931411  0.42612698 -0.38094383 -0.42910295]
 [ 0.88823396 -0.58653639 -1.34931411  0.52256725 -0.38094383 -0.39396181]
 [ 0.41643054 -0.58653639 -1.34931411  0.52256725 -0.38094383 -0.3764894 ]
 [ 0.09823754 -0.58653639 -1.4301041   0.52256725 -0.38094383 -0.35901698]
 [ 0.05434885 -0.58653639 -1.4301041   0.61900753 -0.38094383 -0.32387584]
 [ 0.26282012 -0.58653639 -1.34931411  0.7154478  -0.38094383 -0.2887347 ]
 [ 0.21893143 -0.65625711 -1.4301041   0.7154478  -0.38094383 -0.25359356]
 [ 0.3505975  -0.58653639 -1.34931411  0.81188808 -0.38094383 -0.21845241]
 [ 0.43837488 -0.58653639 -1.34931411  0.90832835 -0.38094383 -0.15700449]
 [ 0.57004095 -0.65625711 -1.34931411  0.90832835 -0.38094383 -0.09555657]]


Se dividen nuevamente los conjuntos de datos

In [17]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainX, testX = train_test_split(datosX, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testX, valX = train_test_split(testX, test_size=0.33, shuffle=False)

print("Las dimensiones de trainX son: ", trainX.shape)
print("Las dimensiones de testX son: ", testX.shape)
print("Las dimensiones de valX son: ", valX.shape)


Las dimensiones de trainX son:  (30635, 12, 6)
Las dimensiones de testX son:  (8797, 12, 6)
Las dimensiones de valX son:  (4333, 12, 6)


In [18]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainY, testY = train_test_split(datosY, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testY, valY = train_test_split(testY, test_size=0.33, shuffle=False)

print("Las dimensiones de trainY son: ", trainY.shape)
print("Las dimensiones de testY son: ", testY.shape)
print("Las dimensiones de valY son: ", valY.shape)

Las dimensiones de trainY son:  (30635, 1)
Las dimensiones de testY son:  (8797, 1)
Las dimensiones de valY son:  (4333, 1)


Se crean métricas para medir desempeño

In [19]:
import tensorflow.keras.backend as K

def smape(y_true, y_pred):
    """
    Define la función SMAPE (Error Porcentual Absoluto Medio Simétrico).
    """
    summ = K.abs(y_true) + K.abs(y_pred)
    smape_val = K.abs(y_pred - y_true) / summ * 2.0
    return K.mean(smape_val, axis=-1)

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def ia(y_true, y_pred):
    numerator = K.sum(K.abs(y_true - y_pred))
    denominator = K.sum(K.abs(y_true - K.mean(y_true)) + K.abs(y_pred - K.mean(y_true)))
    return 1 - (numerator / denominator)

In [20]:
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_error as mae

def graficarPrediccion(modelo, x, y, inicio, final):
  predicciones = modelo.predict(x)
  predicciones = predicciones.flatten()
  df = pd.DataFrame({'Originales': y, 'Predichos': predicciones})
  plt.plot(df.index, df['Originales'][inicio:final], label='Originales')
  plt.plot(df.index, df['Predichos'][inicio:final], label='Predichos')
  return df, mse(y, predicciones),  mae(y, predicciones), rmse(y, predicciones), smape(y,predicciones), ia(y, predicciones)

Versión Final


In [21]:
def objective(params):

    model = Sequential()
    model.add(InputLayer(input_shape=(trainX.shape[1], trainX.shape[2])))
    if (params['layers'] == 1):
      model.add(GRU(units=params['units'], activation=params['activation'], return_sequences=False))
      model.add(Dropout(params['dropout']))

    else:
      for _ in range(int(params['layers']) - 1):
          model.add(GRU(units=params['units'], activation=params['activation'], return_sequences=True))
          model.add(Dropout(params['dropout']))
      model.add(GRU(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    model.add(Dense(1))


    opt = Adam(learning_rate=params['learning_rate'])
    model.compile(optimizer=opt, loss='mse', metrics=["mae", smape, rmse, ia])

    early_stopping = EarlyStopping(monitor='val_loss', patience=15, verbose=1, restore_best_weights= True)

    model.fit(trainX, trainY, epochs=params['epochs'],
                        validation_data=(testX, testY),
                        verbose = 2, batch_size=params['batch'], callbacks=[early_stopping])


    predictions = model.predict(valX)


    loss = mean_squared_error(valY, predictions)

    return {'loss': loss, 'status': STATUS_OK}

In [22]:
trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=50, trials=trials, rstate=np.random.default_rng(42))

  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                           

958/958 - 61s - 64ms/step - ia: 0.2709 - loss: 0.9545 - mae: 0.7350 - rmse: 0.9563 - smape: 1.4958 - val_ia: 0.2460 - val_loss: 0.8096 - val_mae: 0.6709 - val_rmse: 0.7570 - val_smape: 1.4127

Epoch 2/128                                           

958/958 - 23s - 24ms/step - ia: 0.3115 - loss: 0.8886 - mae: 0.7019 - rmse: 0.9244 - smape: 1.4277 - val_ia: 0.2498 - val_loss: 0.7889 - val_mae: 0.6623 - val_rmse: 0.7517 - val_smape: 1.3544

Epoch 3/128                                           

958/958 - 19s - 20ms/step - ia: 0.3278 - loss: 0.8708 - mae: 0.6940 - rmse: 0.9152 - smape: 1.4021 - val_ia: 0.2491 - val_loss: 0.7961 - val_mae: 0.6679 - val_rmse: 0.7568 - val_smape: 1.3859

Epoch 4/128                                           

958/958 - 32s - 34ms/step - ia: 0.3362 - loss: 0.8622 - mae: 0.6879 - rmse: 0.9099 - smape: 1.3847 - val_ia: 0.2503 - val_loss: 0.7816 - val_mae: 0.6595 - val_rmse: 0.7503 - val_smape: 1.3423

Epoc

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/16                                                                           

120/120 - 52s - 433ms/step - ia: 0.3223 - loss: 0.8546 - mae: 0.6816 - rmse: 0.9212 - smape: 1.3849 - val_ia: 0.3292 - val_loss: 0.8074 - val_mae: 0.6810 - val_rmse: 0.8409 - val_smape: 1.3433

Epoch 2/16                                                                           

120/120 - 11s - 93ms/step - ia: 0.4086 - loss: 0.7937 - mae: 0.6500 - rmse: 0.8880 - smape: 1.2460 - val_ia: 0.3384 - val_loss: 0.7580 - val_mae: 0.6398 - val_rmse: 0.8035 - val_smape: 1.2824

Epoch 3/16                                                                           

120/120 - 13s - 109ms/step - ia: 0.4218 - loss: 0.7768 - mae: 0.6414 - rmse: 0.8793 - smape: 1.2287 - val_ia: 0.3525 - val_loss: 0.8028 - val_mae: 0.6718 - val_rmse: 0.8406 - val_smape: 1.3069

Epoch 4/16                                                                           

120/120 - 15s - 128ms/step - ia: 0.4370 - loss: 0.7601 - mae: 0.6326 - 

...

120/120 - 6s - 51ms/step - ia: 0.3158 - loss: 1.0417 - mae: 0.7505 - rmse: 1.0170 - smape: 1.3767 - val_ia: 0.3034 - val_loss: 0.8873 - val_mae: 0.7052 - val_rmse: 0.8695 - val_smape: 1.4352

Epoch 2/256                                                                         

120/120 - 1s - 8ms/step - ia: 0.3230 - loss: 0.9372 - mae: 0.7171 - rmse: 0.9655 - smape: 1.3797 - val_ia: 0.3129 - val_loss: 0.8229 - val_mae: 0.6784 - val_rmse: 0.8368 - val_smape: 1.3832

Epoch 3/256                                                                         

120/120 - 1s - 7ms/step - ia: 0.3407 - loss: 0.8999 - mae: 0.7039 - rmse: 0.9460 - smape: 1.3608 - val_ia: 0.3202 - val_loss: 0.7962 - val_mae: 0.6648 - val_rmse: 0.8227 - val_smape: 1.3482

Epoch 4/256                                                                         

120/120 - 1s - 11ms/step - ia: 0.3582 - loss: 0.8745 - mae: 0.6909 - rmse: 0.9332 - smape: 1.3329 - val_ia: 0.3228 - val_loss: 0.7793 - val_mae: 0.6551 - val_rmse: 0.81

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/16                                                                          

958/958 - 28s - 29ms/step - ia: 0.2844 - loss: 1.0469 - mae: 0.7551 - rmse: 0.9981 - smape: 1.4409 - val_ia: 0.2488 - val_loss: 0.8187 - val_mae: 0.6639 - val_rmse: 0.7505 - val_smape: 1.3924

Epoch 2/16                                                                          

958/958 - 12s - 12ms/step - ia: 0.3382 - loss: 0.8876 - mae: 0.6975 - rmse: 0.9239 - smape: 1.3528 - val_ia: 0.2554 - val_loss: 0.7846 - val_mae: 0.6490 - val_rmse: 0.7398 - val_smape: 1.2844

Epoch 3/16                                                                          

958/958 - 11s - 12ms/step - ia: 0.3622 - loss: 0.8551 - mae: 0.6815 - rmse: 0.9074 - smape: 1.3101 - val_ia: 0.2564 - val_loss: 0.7777 - val_mae: 0.6509 - val_rmse: 0.7427 - val_smape: 1.2777

Epoch 4/16                                                                          

958/958 - 21s - 22ms/step - ia: 0.3713 - loss: 0.8426 - mae: 0.6768 - rmse: 0

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                         

240/240 - 12s - 50ms/step - ia: 0.2859 - loss: 0.9719 - mae: 0.7370 - rmse: 0.9781 - smape: 1.4358 - val_ia: 0.3005 - val_loss: 0.7832 - val_mae: 0.6514 - val_rmse: 0.7961 - val_smape: 1.2869

Epoch 2/128                                                                         

240/240 - 3s - 11ms/step - ia: 0.3736 - loss: 0.8523 - mae: 0.6837 - rmse: 0.9182 - smape: 1.3106 - val_ia: 0.3171 - val_loss: 0.7623 - val_mae: 0.6412 - val_rmse: 0.7872 - val_smape: 1.2567

Epoch 3/128                                                                         

240/240 - 3s - 12ms/step - ia: 0.3780 - loss: 0.8399 - mae: 0.6771 - rmse: 0.9125 - smape: 1.3048 - val_ia: 0.3253 - val_loss: 0.7577 - val_mae: 0.6382 - val_rmse: 0.7862 - val_smape: 1.2402

Epoch 4/128                                                                         

240/240 - 3s - 12ms/step - ia: 0.3875 - loss: 0.8340 - mae: 0.6735 - rmse: 0.90

In [23]:
print(best)

{'activation': 1, 'batch': 2, 'dropout': 0.2, 'epochs': 4, 'layers': 1.0, 'learning_rate': 0.0002707756079796208, 'units': 4}
